# 1. Guardar el csv en una tabla

In [0]:
# Leer el archivo CSV
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("file:/Workspace/Users/mago-dios666@hotmail.com/modelo_predictivo/data/PL_2025_actual.csv")

# Guardar en la tabla con modo overwrite (reemplazar datos existentes)
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.bronze.actual_premier")

print(f"Tabla actualizada: {df.count()} registros cargados")

# unir tabla actual_premier con historico_premier


In [0]:
# Unificar historico_premier y actual_premier y guardar en silver.consolidado_premier

historico_df = spark.table("workspace.bronze.historico_premier")
actual_df = spark.table("workspace.bronze.actual_premier")

from pyspark.sql.functions import expr

historico_df = historico_df.withColumn("season", expr("try_cast(season as string)"))
actual_df = actual_df.withColumn("season", expr("try_cast(season as string)"))

consolidado_df = historico_df.unionByName(actual_df)

consolidado_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.consolidado_premier")